# F2 — W4 Jun 13 Spike Cluster at 14:00 UTC

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [ ]:
wm = WINDOWS_META['W4']
day = '2025-06-13'
mids = {}
for sym in [wm['front'], wm['back']]:
    fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
    assert fpath, f'Missing {sym} {day}'
    df = pd.read_parquet(fpath[0], columns=['ts_event','bid_px_00','ask_px_00'])
    df['ts_event'] = pd.to_datetime(df['ts_event'])
    mask = (df['ts_event'].dt.hour == 13) & (df['ts_event'].dt.minute >= 55) |            (df['ts_event'].dt.hour == 14) & (df['ts_event'].dt.minute <= 5)
    df = df[mask].set_index('ts_event')
    mids[sym] = ((df['bid_px_00']+df['ask_px_00'])/2).resample('1ms').last().ffill()
    del df; gc.collect()

cal = mids[wm['back']].subtract(mids[wm['front']]).dropna()

# load trade annotations from results
trades = pd.read_parquet(RESULTS_DIR / wm['result_key'] / 'none' / 'trades.parquet')
trades['entry_time'] = pd.to_datetime(trades['entry_time'])
spike_trades = trades[trades['entry_time'].dt.date.astype(str) == day].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(cal.index, cal.values, lw=0.8, color='steelblue')
ax.axvline(pd.Timestamp('2025-06-13 14:00:00'), color='red', ls='--', lw=1.2, label='14:00 UTC')
trade_labels = {6: 'T6 Long SL\n−$12.50', 7: 'T7 Long SL\n−$43.75', 8: 'T8 Long TP\n+$43.75'}
offsets = {6: 0.3, 7: 0.6, 8: 0.9}
for idx, row in spike_trades.iterrows():
    ti = row['entry_time']
    lbl = trade_labels.get(idx, f'T{idx}')
    cal_val = cal.reindex([ti], method='nearest').values[0]
    color = '#2ecc71' if row['gross_usd'] > 0 else '#e74c3c'
    ax.annotate(lbl, xy=(ti, cal_val), xytext=(ti, cal_val + offsets.get(idx, 0.3)),
                fontsize=8, color=color, arrowprops=dict(arrowstyle='->', color=color))
ax.set_xlabel('Time (UTC)'); ax.set_ylabel('Calendar Spread (pts)')
ax.set_title('W4 ESM5→ESU5: 14:00 UTC Spike Cluster — 2025-06-13 (1ms resolution)')
ax.legend()
save_fig(fig, 'F2_w4_jun13_spike_cluster.png')
